In [4]:
from fastmcp.utilities.docstring_parsing import parse_docstring
from langchain_core.tools import Tool, tool
from typing import Literal
from random import randint

from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage
from langgraph.constants import START, END
from langgraph.graph import MessagesState, StateGraph
from langchain_deepseek import ChatDeepSeek

from dotenv import load_dotenv
from pygments.lexers.haskell import LiterateCryptolLexer

# 加载.env文件中的配置
load_dotenv(override=True)

# 定义模型
model = ChatDeepSeek(
    model="deepseek-flash",
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)

# 定义天气工具
@tool(parse_docstring=True)
def get_weather(city:str="上海"):
    """
    查询指定城市当日天气

    Args:
        city: 城市名称
    """
    return f"{city}的天气是晴朗的"


# 定义查询新闻工具
@tool(parse_docstring=True)
def get_news(domain:Literal["AI","食品安全"]):
    """
    查询特定领域的当日热点

    Args:
        domain: 特定领域
    """
    if domain=="AI":
        return "Anthropic发布了 Claude Opus-4.8, 但通过API用中文它你是谁时，大多数情况下返回的是”Qwen“或者”Deepseek“。"
    elif domain=="食品安全":
        return "双汇发展子公司猪肉产品被抽检出抗生素超标37.5倍"
    else:
        return "未知领域新闻"

tools=[get_weather,get_news]
model_with_tools=model.bind_tools(tools=tools)

# 全局状态
class OverAllState(MessagesState):
    user_input:str
    final_output:str

# 输入节点
def input_node(state:OverAllState)->OverAllState:
    return {
        "messages":[HumanMessage(state["user_input"])]
    }

# 大模型节点
def llm_node(state:OverAllState)->OverAllState:
    ai_msg=model_with_tools.invoke(state["messages"])
    return {
        "messages":ai_msg
    }

# 工具节点
def tool_node(state:OverAllState)->OverAllState:
    # 判断大模型返回的信息，是否调用工具
    messages=state["messages"]
    ai_msg=messages[-1]
    tool_calls=ai_msg.tool_calls


    # 表示调用工具失败的概率
    fail_prob = 6
    # 循环取出要调用的工具名称
    for tool_call in tool_calls:
        if tool_call["name"] == "get_weather":

            if randint(0,9)<fail_prob:
                messages.append(ToolMessage(
                    content="网络拨动，调用失败，请重试",
                    tool_call_id=tool_call["id"],
                ))
            else:
                messages.append(get_weather.invoke(tool_call))
        elif tool_call["name"] == "get_news":

            if randint(0,9)<fail_prob:
                messages.append(ToolMessage(
                    content="网络拨动，调用失败，请重试",
                    tool_call_id=tool_call["id"],
                ))
            else:
                messages.append(get_news.invoke(tool_call))
        else:
             messages.append(ToolMessage(
                    content="工具名称错误，调用失败，请重试",
                    tool_call_id=tool_call["id"],
                ))
    return {
        "messages":messages
    }

# 输出节点
def output_node(state:OverAllState)->OverAllState:
    return {
        "final_output":state["messages"][-1].content,
    }

# 路由节点
def router(state:OverAllState)->Literal["tool_node","output_node"]:
    # 判断消息列表中是否有工具调用请求信息
    messages=state["messages"]
    last_msg=messages[-1]
    if last_msg.tool_calls:
        return "tool_node"
    return "output_node"

builder = StateGraph(state_schema=OverAllState)
builder.add_node("input_node",input_node)
builder.add_node("llm_node",llm_node)
builder.add_node("tool_node",tool_node)
builder.add_node("output_node",output_node)

builder.add_edge(START,"input_node")
builder.add_edge("input_node","llm_node")
builder.add_conditional_edges("llm_node",router)
builder.add_edge("tool_node","llm_node")
builder.add_edge("output_node",END)

graph = builder.compile()
res=graph.invoke({
    "user_input":"查询今天的上海天气和AI新闻热点",
    "messages":[SystemMessage("如果工具调用失败，必须重新调用直到成功为止")]
})

print("user_input:",res["user_input"])
print("final_output:",res["final_output"])
for msg in res["messages"]:
    msg.pretty_print()



{'messages': [SystemMessage(content='如果工具调用失败，必须重新调用直到成功为止', additional_kwargs={}, response_metadata={}, id='3839fb58-6f2f-4831-9885-42c1bb7d6135'), HumanMessage(content='查询今天的上海天气和AI新闻热点', additional_kwargs={}, response_metadata={}, id='700be1a0-1040-4248-b217-8d189dfdaa4a'), AIMessage(content='我来为您查询今天上海的天气和AI新闻热点。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 353, 'total_tokens': 430, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 128}, 'prompt_cache_hit_tokens': 128, 'prompt_cache_miss_tokens': 225}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501', 'id': '1980021d-1dcd-45c8-be7b-8455e3f3ed53', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a14cd9-6ed5-7563-85ad-8da2199e7c12-0', tool_calls=[{'name': 'get_weather', 'args': {'city': '上海'}, 'id': 'call_00_M1dCRdQ3LHJeInuMRpYx871